# 04 — Modelado y Validación Cruzada
**Fase 4 · CRISP-DM** | Proyecto: Predicción de Stroke

UNAM · Facultad de Ingeniería · Minería de Datos · Semestre 2027-1

---

Modelos evaluados:
- **Regresión Logística** — baseline interpretable (lineal)
- **Random Forest** — ensamble por bagging (robusto, captura no-linealidades)
- **XGBoost** — Extreme Gradient Boosting (árboles secuenciales optimizados con regularización)
- **LightGBM** — Light Gradient Boosting Machine (crecimiento leaf-wise, alta eficiencia y velocidad)

Validación: `StratifiedKFold` con **k=5** sobre el conjunto de entrenamiento **sin balancear**. SMOTE se aplica dentro de cada fold mediante `imblearn.pipeline.Pipeline`, de modo que solo se sobremuestrean los folds de entrenamiento y cada fold de validación conserva la distribución real de clases (≈4.9 % positivos).  
El ajuste final usa el conjunto completo balanceado con SMOTE (`processed_train.csv`).  
Los modelos entrenados se serializan en `data/`.

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, roc_auc_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

DATA_DIR     = os.path.join(os.getcwd(), "data")
RANDOM_STATE = 42
CV_FOLDS     = 5

# Cargar conjunto de entrenamiento procesado (post-SMOTE)
train_df = pd.read_csv(os.path.join(DATA_DIR, "processed_train.csv"))
X_train  = train_df.drop(columns=["stroke"])
y_train  = train_df["stroke"]
print(f"Train cargado: {X_train.shape[0]} muestras · {X_train.shape[1]} features")

# Conjunto de entrenamiento sin balancear (pre-SMOTE) para la validación cruzada
train_unbal_df = pd.read_csv(os.path.join(DATA_DIR, "processed_train_unbalanced.csv"))
X_train_cv     = train_unbal_df.drop(columns=["stroke"])
y_train_cv     = train_unbal_df["stroke"]
print(f"Train sin balancear (CV): {X_train_cv.shape[0]} muestras · "
      f"distribución {y_train_cv.value_counts().to_dict()}")

Train cargado: 7776 muestras · 20 features
Train sin balancear (CV): 4087 muestras · distribución {0: 3888, 1: 199}


## 4.1 Definición de modelos

In [2]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        solver="lbfgs",
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=5,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        n_jobs=-1,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=5,
        random_state=RANDOM_STATE,
        verbose=-1,
        n_jobs=-1,
    ),
}
print("Modelos definidos:", list(models.keys()))

Modelos definidos: ['Logistic Regression', 'Random Forest', 'XGBoost', 'LightGBM']


## 4.2 Validación cruzada estratificada (5-fold)
SMOTE va dentro del `Pipeline`: en cada iteración se ajusta solo con el fold de entrenamiento, y las métricas se calculan sobre un fold de validación sin muestras sintéticas.

In [3]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "auc_roc": "roc_auc",
    "recall":  make_scorer(recall_score),
    "f1":      make_scorer(f1_score),
}

cv_results = []
for name, model in models.items():
    print(f"Validando: {name} ...")
    pipe = ImbPipeline([
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", model),
    ])
    res = cross_validate(pipe, X_train_cv, y_train_cv, cv=cv, scoring=scoring,
                         return_train_score=False, n_jobs=-1)
    cv_results.append({
        "Modelo":         name,
        "AUC-ROC (mean)": round(float(np.mean(res["test_auc_roc"])), 4),
        "AUC-ROC (std)":  round(float(np.std(res["test_auc_roc"])),  4),
        "Recall (mean)":  round(float(np.mean(res["test_recall"])),  4),
        "Recall (std)":   round(float(np.std(res["test_recall"])),   4),
        "F1 (mean)":      round(float(np.mean(res["test_f1"])),      4),
    })

cv_df = pd.DataFrame(cv_results)
cv_df

Validando: Logistic Regression ...


Validando: Random Forest ...


Validando: XGBoost ...


Validando: LightGBM ...


,Modelo,AUC-ROC (mean),AUC-ROC (std),Recall (mean),Recall (std),F1 (mean)
0,Logistic Regression,0.8007,0.0433,0.4022,0.0602,0.2306
1,Random Forest,0.7919,0.0243,0.1105,0.0253,0.1055
2,XGBoost,0.7911,0.0342,0.3118,0.0361,0.2027
3,LightGBM,0.7934,0.0346,0.3018,0.0380,0.2065


## 4.3 Entrenamiento final y serialización

In [4]:
for name, model in models.items():
    model.fit(X_train, y_train)
    fname = name.lower().replace(" ", "_").replace("-", "")
    path  = os.path.join(DATA_DIR, f"model_{fname}.pkl")
    with open(path, "wb") as f:
        pickle.dump(model, f)
    print(f"Modelo guardado: {path}")

Modelo guardado: C:\PROYECTOS LATEX y otros\MIneria de datos\PROYECTO\CODIGO\data\model_logistic_regression.pkl


Modelo guardado: C:\PROYECTOS LATEX y otros\MIneria de datos\PROYECTO\CODIGO\data\model_random_forest.pkl


Modelo guardado: C:\PROYECTOS LATEX y otros\MIneria de datos\PROYECTO\CODIGO\data\model_xgboost.pkl


Modelo guardado: C:\PROYECTOS LATEX y otros\MIneria de datos\PROYECTO\CODIGO\data\model_lightgbm.pkl


---
✅ **Fase de modelado completada.**  
Continúa en `05_evaluation.ipynb`.